# VitaStress — Figura 4
Riproduzione dei violin plot del paper su tutti i 21 soggetti.

## Fase 0 — Setup
Librerie e download del dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

if not os.path.exists("VitaStress"):
    !git clone https://github.com/paulvincenz/VitaStress.git

DATA = "VitaStress/data"

## Fase 1 — Persone
Elenco dei 21 soggetti.

In [ ]:
persone = sorted([c.replace("id_", "") for c in os.listdir(DATA) if c.startswith("id_")])
print("Persone:", len(persone))

## Fase 2 — Orari delle fasi
Dalle annotazioni ricava inizio e fine di ogni fase. Salta le fasi mancanti o con durata impossibile.

In [ ]:
fasi = {"baseline": "baseline", "cognitive": "cognitive",
        "public": "public speaking", "physical": "physical"}

def trova_orari(persona):
    cartella = DATA + "/id_" + persona + "/" + persona + "_"
    ann = pd.read_csv(cartella + "annotation.csv")
    ann["timestamp"] = pd.to_datetime(ann["timestamp"], format="mixed")
    ann["evento"] = ann["Button Name"].astype(str).str.lower().str.replace(":", "").str.strip(" []'")
    orari = {}
    for fase in fasi:
        nome = fasi[fase]
        inizio = ann[ann["evento"].str.startswith(nome + " start")]["timestamp"]
        fine = ann[ann["evento"].str.startswith(nome + " stop")]["timestamp"]
        if len(inizio) == 0 or len(fine) == 0:
            print(persona[:8], fase, "-> manca inizio o fine, salto")
            continue
        durata = (fine.iloc[0] - inizio.iloc[0]).total_seconds() / 60
        if durata <= 0 or durata > 30:
            print(persona[:8], fase, "-> durata strana:", round(durata, 1), "min, salto")
            continue
        orari[fase] = (inizio.iloc[0], fine.iloc[0])
    return orari

## Fase 3 — Tagliare ed etichettare
Tiene solo i valori dentro gli orari di ogni fase e ci scrive accanto la fase.

In [ ]:
def taglia(dati, colonna, orari):
    risultato = []
    for fase in orari:
        inizio, fine = orari[fase]
        pezzo = dati[(dati["date"] >= inizio) & (dati["date"] <= fine)].copy()
        pezzo["fase"] = fase
        risultato.append(pezzo[[colonna, "fase"]])
    return pd.concat(risultato, ignore_index=True)

## Fase 4 — Tutte le persone, tutte le misure
Pulizia, media dell'accelerometro ogni 30 s, standardizzazione per persona.

In [ ]:
variabili = [("activity", "bpm"), ("emography", "stress_skin"), ("temperature", "temp_sk1"),
             ("acc", "accX"), ("acc", "accY"), ("acc", "accZ")]

tutte = []
for persona in persone:
    orari = trova_orari(persona)
    cartella = DATA + "/id_" + persona + "/" + persona + "_"
    for file, colonna in variabili:
        dati = pd.read_csv(cartella + file + ".csv", usecols=["date", colonna])
        dati["date"] = pd.to_datetime(dati["date"], format="mixed")
        if file == "acc":
            dati = dati.resample("30s", on="date").mean().reset_index()
        tab = taglia(dati, colonna, orari)
        tab = tab.rename(columns={colonna: "valore"})
        if colonna == "bpm" or colonna == "temp_sk1":
            tab["valore"] = tab["valore"].replace(0, np.nan)
        tab["grezzo"] = tab["valore"]
        tab["valore"] = (tab["valore"] - tab["valore"].mean()) / tab["valore"].std()
        tab["persona"] = persona
        tab["variabile"] = colonna
        tutte.append(tab)

tutte = pd.concat(tutte, ignore_index=True)
print("Righe totali:", len(tutte))

## Fase 5 — Figura 4
Un riquadro per misura, un violino per fase.

In [ ]:
ordine = ["baseline", "cognitive", "physical", "public"]

plt.figure(figsize=(15, 8))
for i, (file, colonna) in enumerate(variabili):
    plt.subplot(2, 3, i + 1)
    sns.violinplot(data=tutte[tutte["variabile"] == colonna], x="fase", y="valore",
                   cut=0, order=ordine)
    plt.title(colonna)
plt.tight_layout()
plt.show()

## Fase 6 — Confronto con il paper
Mediane dei valori originali e zeri del sudore.

In [ ]:
print(tutte.groupby(["variabile", "fase"])["grezzo"].median().unstack()[ordine].round(2))

sudore = tutte[tutte["variabile"] == "stress_skin"]
print("\n% di zeri nel sudore:")
print((sudore.groupby("fase")["grezzo"].apply(lambda s: (s == 0).mean() * 100)).round(0))

## Note
- Battito 0 e temperatura 0 °C → dati mancanti (errore del sensore)
- `stress_skin`: gli zeri sono valori veri
- Persona `a5e823ad`: riposo di 82 min escluso
- Accelerometro: media ogni 30 s
- Standardizzazione per persona: appiattisce il sudore (molti zeri)
- Temperatura: andamento del paper non riprodotto